Judge saved model responses and record case-level scores, criterion decisions and token usage.


In [ ]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'evaluations/human_and_model_validation/opus5_judging/judge_opus5'
os.chdir(_NOTEBOOK_DIR)


In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any, Dict, List, Tuple
import pandas as pd
from tqdm.auto import tqdm


In [ ]:
MODEL = 'claude-opus-5'
CLAUDE_BIN = shutil.which('claude') or 'claude'
MAX_WORKERS = 4
REQUEST_TIMEOUT = 600
BASE_SLEEP_SECONDS = 2
MAX_RETRIES = 6
N_CASES = 60
PROTOCOL_COLUMNS = [f'protocol_{i}' for i in range(1, 5)]
INPUT_CSV = '../../blind_protocols_60.csv'
KEY_CSV = '../../blind_protocols_60_key.csv'
OUTPUT_CSV = 'blind_protocols_60_opus5_judge.csv'
STATS_CSV = 'blind_protocols_60_opus5_stats_by_source.csv'
CACHE_JSONL = 'judge_cache_opus5.jsonl'

def parse_criteria(criterion_text: str) -> List[Dict[str, Any]]:
    if not criterion_text or pd.isna(criterion_text):
        return []
    criteria: List[Dict[str, Any]] = []
    blocks = re.split('\\n?\\[(\\d+)\\]\\n', str(criterion_text).strip())
    for i in range(1, len(blocks), 2):
        crit_id = int(blocks[i])
        body = blocks[i + 1] if i + 1 < len(blocks) else ''
        points_m = re.search('points:\\s*(-?\\d+)', body)
        tags_m = re.search('tags:\\s*\\[(.+?)\\]', body)
        crit_m = re.search('criterion:\\s*\\n([\\s\\S]+)', body)
        points = int(points_m.group(1)) if points_m else 0
        tags = [t.strip().strip('\'"') for t in tags_m.group(1).split(',')] if tags_m else []
        crit_text = crit_m.group(1).strip() if crit_m else ''
        if crit_text:
            criteria.append({'id': crit_id, 'points': points, 'tags': tags, 'criterion': crit_text})
    return criteria

def load_dataset(path: str) -> pd.DataFrame:
    d = pd.read_csv(path, sep=';', encoding='utf-8-sig', low_memory=False)
    d.columns = [c.lstrip('\ufeff') for c in d.columns]
    if N_CASES:
        d = d.head(N_CASES)
    missing = [c for c in ['case_id', 'case_text', 'Criterion', *PROTOCOL_COLUMNS] if c not in d.columns]
    if missing:
        raise ValueError(f'Required columns missing in {path}: {missing}')
    if not d['case_id'].is_unique:
        raise ValueError('case_id must be unique in the blind dataset')
    id_columns = [c for c in d.columns if c not in PROTOCOL_COLUMNS]
    long = d.melt(id_vars=id_columns, value_vars=PROTOCOL_COLUMNS, var_name='protocol_column', value_name='winner_protocol')
    long['blind_slot'] = long['protocol_column'].str.extract('(\\d+)$').astype(int)
    long = long.sort_values(['case_id', 'blind_slot'], kind='stable').reset_index(drop=True)
    if long['winner_protocol'].isna().any() or long['winner_protocol'].astype(str).str.strip().eq('').any():
        raise ValueError('The blind dataset contains an empty protocol')
    if len(long) != len(d) * len(PROTOCOL_COLUMNS):
        raise AssertionError('Unexpected number of unfolded protocols')
    return long
preview_df = load_dataset(INPUT_CSV)
print(f'Judge model: {MODEL} via {CLAUDE_BIN}')
print(f"Blind cases: {preview_df['case_id'].nunique()}")
print(f'Protocols to judge: {len(preview_df)}')
print(f'Output: {OUTPUT_CSV}')


In [ ]:
_RESPONSE_SCHEMA = {'type': 'object', 'properties': {'judgments': {'type': 'array', 'items': {'type': 'object', 'properties': {'id': {'type': 'integer'}, 'explanation': {'type': 'string'}, 'criteria_met': {'type': 'boolean'}}, 'required': ['id', 'explanation', 'criteria_met'], 'additionalProperties': False}}}, 'required': ['judgments'], 'additionalProperties': False}
judge_cache: Dict[str, Dict[str, Any]] = {}

def _make_cache_key(system_text: str, user_text: str) -> str:
    raw = json.dumps({'model': MODEL, 'system': system_text, 'user': user_text}, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def load_disk_cache(path: str=CACHE_JSONL) -> int:
    if not os.path.exists(path):
        return 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            judge_cache[rec['key']] = rec['value']
    return len(judge_cache)

def _append_disk_cache(key: str, value: Dict[str, Any], path: str=CACHE_JSONL) -> None:
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps({'key': key, 'value': value}, ensure_ascii=False) + '\n')

def _agent_argv(system_text: str) -> List[str]:
    return [CLAUDE_BIN, '-p', '--model', MODEL, '--system-prompt', system_text, '--json-schema', json.dumps(_RESPONSE_SCHEMA, ensure_ascii=False), '--tools', '', '--output-format', 'json', '--no-session-persistence', '--disable-slash-commands', '--strict-mcp-config', '--permission-prompts', 'none', '--safe-mode']

def call_judge(system_text: str, user_text: str) -> Dict[str, Any]:
    cache_key = _make_cache_key(system_text, user_text)
    if cache_key in judge_cache:
        cached = judge_cache[cache_key]
        return {'text': cached['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cost_usd': 0.0, 'cached': True}
    last_error = ''
    for attempt in range(MAX_RETRIES):
        try:
            proc = subprocess.run(_agent_argv(system_text), input=user_text.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=REQUEST_TIMEOUT)
            if proc.returncode != 0:
                raise RuntimeError(f"claude exited {proc.returncode}: {proc.stderr.decode('utf-8', 'replace')[:400]} | {proc.stdout.decode('utf-8', 'replace')[:400]}")
            payload = json.loads(proc.stdout.decode('utf-8', 'replace'))
            if payload.get('is_error'):
                raise RuntimeError(f"agent error: {str(payload.get('result'))[:400]}")
            structured = payload.get('structured_output')
            text = json.dumps(structured, ensure_ascii=False) if structured else str(payload.get('result') or '')
            if not text.strip():
                raise RuntimeError('empty agent response')
            usage = payload.get('usage') or {}
            prompt_tokens = int((usage.get('input_tokens') or 0) + (usage.get('cache_creation_input_tokens') or 0) + (usage.get('cache_read_input_tokens') or 0))
            completion_tokens = int(usage.get('output_tokens') or 0)
            cost = float(payload.get('total_cost_usd') or 0.0)
            value = {'text': text, 'prompt_tokens': prompt_tokens, 'completion_tokens': completion_tokens, 'total_tokens': prompt_tokens + completion_tokens, 'cost_usd': cost}
            judge_cache[cache_key] = value
            _append_disk_cache(cache_key, value)
            return {**value, 'cached': False}
        except Exception as e:
            last_error = str(e)
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
    raise RuntimeError(f'Judge call failed after {MAX_RETRIES} retries. Last error: {last_error}')
print('restored from disk cache:', load_disk_cache())


In [ ]:
_JUDGE_SYSTEM_PROMPT = 'You are an expert medical protocol evaluator.\nYour task is to assess a clinical protocol written in response to a clinical case\nagainst each of the specified criteria.\n\nBase your judgment solely on the content of the protocol and the clinical case. Do not reward content that is not explicitly present in the protocol.\nFor positive criteria (described as what should be done): return criteria_met=true if the protocol satisfies the criterion.\nFor penalty criteria: return criteria_met=true if the protocol commits the described failure.\nJudge every criterion independently of the others.\nReturn JSON only: an object with a field "judgments" holding one entry per criterion, in the given order, each with fields "id" (integer), "explanation" (one concise sentence) and "criteria_met" (boolean).'

def _strip_author(protocol_text: str) -> str:
    if not protocol_text:
        return ''
    parts = re.split('(?im)^\\s*Protocol Author\\s*:', str(protocol_text), maxsplit=1)
    return parts[0].rstrip()

def _criterion_block(criterion: Dict[str, Any]) -> str:
    direction = 'Positive criterion (met = protocol satisfies it)' if criterion['points'] > 0 else 'Penalty criterion (met = protocol commits this failure)'
    return f"Criterion [{criterion['id']}] — {direction}:\n{criterion['criterion']}"

def build_protocol_user_prompt(case_text, winner_protocol, criteria) -> str:
    blocks = '\n\n'.join((_criterion_block(c) for c in criteria))
    return f'Clinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nProtocol:\n<<<PROTOCOL>>>\n{_strip_author(winner_protocol)}\n<<<END PROTOCOL>>>\n\nCriteria to evaluate ({len(criteria)} total, judge each independently):\n{blocks}\n'.strip()

def parse_judge_output(text, criteria) -> List[Dict[str, Any]]:
    raw = str(text).strip() if text else ''
    data = None
    try:
        data = json.loads(raw)
    except Exception:
        m = re.search('\\{[\\s\\S]*\\}', raw)
        if m:
            try:
                data = json.loads(m.group(0))
            except Exception:
                data = None
    if not isinstance(data, dict) or not isinstance(data.get('judgments'), list):
        raise ValueError(f'Could not parse judge output: {raw[:200]}')
    by_id = {}
    for item in data['judgments']:
        if not isinstance(item, dict) or 'criteria_met' not in item:
            continue
        by_id[int(item.get('id', -1))] = {'criteria_met': bool(item['criteria_met']), 'explanation': str(item.get('explanation', '')).strip()}
    missing = [c['id'] for c in criteria if c['id'] not in by_id]
    if missing:
        raise ValueError(f'Judge returned no verdict for criteria {missing}')
    return [by_id[c['id']] for c in criteria]


In [ ]:
def empty_uni_result() -> Dict[str, Any]:
    return {'uni_details': '[]', 'uni_prompt_tokens': 0, 'uni_completion_tokens': 0, 'uni_total_tokens': 0, 'uni_cached_calls': 0}
print('Universal criteria: disabled')
print('Compatibility columns:', ', '.join(empty_uni_result().keys()))


In [ ]:
def _build_judge_analysis(judgment_records):
    met_pos = [j for j in judgment_records if j['met'] and j['points'] > 0]
    miss_pos = [j for j in judgment_records if not j['met'] and j['points'] > 0]
    triggered = [j for j in judgment_records if j['met'] and j['points'] < 0]
    lines = []
    if met_pos:
        lines.append('Satisfied criteria:')
        for j in met_pos:
            lines.append(f"  [{j['id']}] +{j['points']} pts — {j['explanation']}")
    if miss_pos:
        lines.append('Missed criteria:')
        for j in miss_pos:
            lines.append(f"  [{j['id']}] +{j['points']} pts — {j['explanation']}")
    if triggered:
        lines.append('Triggered penalties:')
        for j in triggered:
            lines.append(f"  [{j['id']}] {j['points']} pts — {j['explanation']}")
    return '\n'.join(lines)

def run_judge_for_case(case_text, winner_protocol, criteria):
    if not criteria:
        return {'Judge_score': None, 'Judge_analysis': '', 'judge_earned': None, 'judge_penalties': None, 'judge_max': None, 'judge_details': '[]', 'judge_prompt_tokens': 0, 'judge_completion_tokens': 0, 'judge_total_tokens': 0, 'judge_cached_calls': 0, 'judge_cost_usd': 0.0}
    user_text = build_protocol_user_prompt(case_text, winner_protocol, criteria)
    response = call_judge(_JUDGE_SYSTEM_PROMPT, user_text)
    parsed = parse_judge_output(response['text'], criteria)
    judgment_records = []
    for criterion, verdict in zip(criteria, parsed):
        judgment_records.append({'id': criterion['id'], 'points': criterion['points'], 'tags': criterion['tags'], 'met': verdict['criteria_met'], 'explanation': verdict['explanation']})
    max_positive = sum((j['points'] for j in judgment_records if j['points'] > 0))
    earned_positive = sum((j['points'] for j in judgment_records if j['met'] and j['points'] > 0))
    triggered_pen = sum((abs(j['points']) for j in judgment_records if j['met'] and j['points'] < 0))
    judge_score = round((earned_positive - triggered_pen) / max_positive, 4) if max_positive > 0 else None
    return {'Judge_score': judge_score, 'Judge_analysis': _build_judge_analysis(judgment_records), 'judge_earned': earned_positive, 'judge_penalties': triggered_pen, 'judge_max': max_positive, 'judge_details': json.dumps(judgment_records, ensure_ascii=False), 'judge_prompt_tokens': response['prompt_tokens'], 'judge_completion_tokens': response['completion_tokens'], 'judge_total_tokens': response['total_tokens'], 'judge_cached_calls': int(response['cached']), 'judge_cost_usd': response['cost_usd']}


In [ ]:
def _empty_row(row_data: Dict[str, Any]) -> Dict[str, Any]:
    for k in ('Judge_score', 'Judge_analysis', 'judge_earned', 'judge_penalties', 'judge_max', 'judge_details'):
        row_data[k] = None
    for k in ('judge_prompt_tokens', 'judge_completion_tokens', 'judge_total_tokens', 'judge_cached_calls'):
        row_data[k] = 0
    row_data['judge_cost_usd'] = 0.0
    row_data.update(empty_uni_result())
    return row_data

def _judge_one(index: int, row_data: Dict[str, Any]) -> Tuple[int, Dict[str, Any]]:
    winner_prot = row_data.get('winner_protocol', '')
    if pd.isna(winner_prot) or not str(winner_prot).strip():
        return (index, _empty_row(row_data))
    criteria = parse_criteria(row_data.get('Criterion', ''))
    result = run_judge_for_case(row_data.get('case_text', ''), winner_prot, criteria)
    row_data.update(result)
    row_data.update(empty_uni_result())
    return (index, row_data)

def run_judge_pipeline(input_df: pd.DataFrame) -> pd.DataFrame:
    tasks = [(i, row.to_dict()) for i, (_, row) in enumerate(input_df.iterrows())]
    results: Dict[int, Dict[str, Any]] = {}
    failures: List[Tuple[int, str]] = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(_judge_one, i, d): i for i, d in tasks}
        for future in tqdm(as_completed(futures), total=len(futures), desc='Judge'):
            i = futures[future]
            try:
                index, row_data = future.result()
                results[index] = row_data
            except Exception as e:
                failures.append((i, str(e)))
                results[i] = _empty_row(dict(tasks[i][1]))
    if failures:
        print(f'WARNING: {len(failures)} protocols failed:')
        for i, err in failures[:10]:
            print(f'  row {i}: {err[:200]}')
    return pd.DataFrame([results[i] for i in range(len(tasks))])

def attach_protocol_sources(result_df: pd.DataFrame, key_csv: str) -> pd.DataFrame:
    key = pd.read_csv(key_csv, sep=';', encoding='utf-8-sig', low_memory=False)
    source_columns = [f'source_of_{i}' for i in range(1, 5)]
    missing = [c for c in ['case_id', *source_columns] if c not in key.columns]
    if missing:
        raise ValueError(f'Required key columns missing: {missing}')
    source_key = key[['case_id', *source_columns]].melt(id_vars='case_id', value_vars=source_columns, var_name='source_column', value_name='protocol_source')
    source_key['blind_slot'] = source_key['source_column'].str.extract('(\\d+)$').astype(int)
    source_key = source_key.drop(columns='source_column')
    merged = result_df.merge(source_key, on=['case_id', 'blind_slot'], how='left', validate='one_to_one')
    if merged['protocol_source'].isna().any():
        raise ValueError('Some evaluated protocols could not be mapped to a source')
    return merged

def score_statistics(result_df: pd.DataFrame) -> pd.DataFrame:
    stats = result_df.groupby('protocol_source', sort=False)['Judge_score'].agg(n='count', mean='mean', std='std', median='median', min='min', max='max').reset_index()
    for column in ['mean', 'std', 'median', 'min', 'max']:
        stats[column] = stats[column].round(4)
    return stats

def run_one(input_csv: str, output_csv: str, stats_csv: str) -> None:
    print(f'\n=== {input_csv}  ->  {output_csv} ===')
    df = load_dataset(input_csv)
    print(f"  cases: {df['case_id'].nunique()} | protocols: {len(df)} | workers: {MAX_WORKERS}")
    judged = run_judge_pipeline(df)
    judged = attach_protocol_sources(judged, KEY_CSV)
    judged.to_csv(output_csv, sep=';', index=False, encoding='utf-8-sig')
    stats = score_statistics(judged)
    stats.to_csv(stats_csv, sep=';', index=False, encoding='utf-8-sig')
    print('\nJudge_score by protocol source:')
    print(stats.to_string(index=False))
    total_tokens = int(judged['judge_total_tokens'].sum() + judged['uni_total_tokens'].sum())
    print(f'\n  total_tokens: {total_tokens:,}')
    print(f"  total_cost_usd: {judged['judge_cost_usd'].sum():.2f}")
    print(f'  saved judgments: {output_csv} ({len(judged)} rows)')
    print(f'  saved statistics: {stats_csv}')
RUN_ALL = True
if RUN_ALL:
    run_one(INPUT_CSV, OUTPUT_CSV, STATS_CSV)
else:
    print('\nRUN_ALL=False - proverka zagruzki vypolnena; pereklyuchite v True dlya progona.')
